# Scaling up: $N=30$ Coupled Classical Oscillators via `paulikit`

This notebook is a companion to
[`N_coupled_harmonic_oscillators_1_D_N_2.ipynb`](N_coupled_harmonic_oscillators_1_D_N_2.ipynb),
which develops the theory (state encoding, Pauli decomposition,
Suzuki-Trotter evolution, Trotter-error bounds, the Hadamard test) on
the $N=2$ case. Section references written "N=2 §x.y" point there.

The $N=2$ notebook builds the Hamiltonian symbolically in SymPy and
Pauli-decomposes it by brute force - for each of the $4^n$ Pauli
labels it builds the $2^n\times2^n$ tensor product and a full matrix
product before taking the trace, $O(32^n)$ in total - which stops
scaling well before $N=30$ (this module's README puts the practical
ceiling at roughly $N=4$). It then runs the circuit through Classiq's
cloud `synthesize`/`execute`. This notebook redoes the Hamiltonian,
decomposition, and evolution steps without either dependency:

1. **The Hamiltonian** is built numerically via
   [`paulikit`](https://pypi.org/project/paulikit/)'s
   `build_hamiltonian` instead of SymPy - SymPy's own symbolic engine
   is part of what caps $N\approx4$, independent of how the
   decomposition is done.
2. **The decomposition** uses `paulikit`'s Fast Walsh-Hadamard
   Transform ($O(n\cdot4^n)$) instead of the brute-force trace loop.
3. **The Trotter-repetition bound** (`alpha_comm`, Proposition 16 of
   Childs, Su, Tran, Wiebe, Zhu, "Theory of Trotter Error with
   Commutator Scaling", PRX 11, 011020 (2021)) is replaced with a
   fully closed-form,
   coefficient-only upper bound - the exact version needs a
   per-term operator-2-norm (an SVD) that does not scale past a few
   hundred Pauli terms, let alone $N=30$'s thousands.
4. **The Trotterized evolution itself** is run with the
   repository's `trotterkit` statevector simulator (§5 below)
   instead of Classiq's cloud `synthesize`/`execute` - a real
   engineering decision, not a drop-in substitute, made explicit
   there.

All four replacements are independently verified against the already-
confirmed $N=2$ results before being trusted at $N=30$.

## Prerequisites

Before working through this notebook, you should be familiar with:

- The $N=2$ notebook, in particular its §4 (Pauli decomposition),
  §5 (Suzuki-Trotter decomposition) and §6.6.1 (Trotter-error bounds)
- The Pauli group: $P^2=I$, and that every Pauli string is unitary
- Basic NumPy and SciPy sparse matrices

## Learning Objectives

By the end of this notebook, you will be able to:

1. Build and Pauli-decompose an $N$-oscillator Hamiltonian numerically
   with `paulikit`, and verify every coefficient by direct projection.
2. Derive a coefficient-only upper bound on the 2nd-order Trotter
   error and turn it into a repetition count.
3. Apply a Pauli-string exponential to a statevector in $O(\dim)$,
   without building any $2^n\times2^n$ matrix.
4. Run the Trotterized evolution with `trotterkit`, verified against
   exact evolution, with a progress tracker on the long loop.

## File Paths and Working Directory

This notebook can be run from any directory. It needs the
`trotterkit` package, which `./bootstrap` (or `make trotterkit`)
builds into the environment from the repository root.

## 1. Physical setup: $N=30$ coupled oscillators

Two conventions are offered, both producing a valid
$(N+M)\times(N+M)$ Hamiltonian via the same `build_hamiltonian` call:

- **Deterministic, nearest-neighbor coupling** (default below): unit
  mass at every oscillator, unit spring constant between adjacent
  oscillators only, zero elsewhere - the simplest standard 1D chain,
  easy to reason about by hand.
- **Seeded random couplings**: every allowed $(i,j)$ pair (on-site and
  coupling) gets an independent random spring constant, masses randomized
  too - exercises the pipeline on a generic coupling network rather
  than one special case, with a fixed seed so results are
  reproducible run to run. Full coupling is far more expensive to
  evolve: $107{,}008$ Pauli terms instead of $14{,}080$, and the
  closed-form bound in §3 asks for $12{,}353$ repetitions at $t=1.0$
  instead of $625$ - about 15 minutes for §6 with `trotterkit`
  (about 73 ms per repetition, measured on this machine). Every
  check through §5 still runs in seconds, except §5's check, which
  needs about 5 minutes here; for §6, set `t_demo = 0.1` ($391$
  repetitions, about 30 seconds).

Both are built as a `scipy.sparse` matrix (`sparse=True`): $N=30$ pads
to $9$ qubits ($\dim=512$), small enough that dense would also work
here, but sparse construction is what lets this same code reach far
larger $N$ without a rewrite: the `paulikit` paper
([arXiv:2610.08299](https://arxiv.org/abs/2610.08299)) runs this same
coupled-oscillator family up to $1{,}470{,}038{,}016$ surviving
terms, with every coefficient checked by exhaustive projection up to
the $91{,}652{,}096$-term $N=150$ case.

In [ ]:
####################################################
# N=30 physical parameters: deterministic or random
#  - build_nearest_neighbor_params(n)
#  - build_random_params(n, seed)
####################################################

import numpy as np

N30 = 30


def build_nearest_neighbor_params(n):
    """Unit mass, unit nearest-neighbor spring constant, zero
    elsewhere - the simplest standard 1D coupled-oscillator chain.
    Every (i, j) pair with i <= j must be present (build_hamiltonian
    requires it), so non-adjacent pairs are set to 0.0 explicitly.
    """
    masses = [1.0] * n
    spring_constants = {}
    for i in range(n):
        for jj in range(i, n):
            spring_constants[(i, jj)] = 1.0 if jj == i + 1 else 0.0
    return masses, spring_constants


def build_random_params(n, seed):
    """Independent random spring constants for every allowed (i, j)
    pair and random positive masses, seeded for reproducibility.
    Spring constants are drawn as abs(normal) so the Hamiltonian stays
    a physically sensible (non-negative stiffness) coupling network.
    """
    rng = np.random.default_rng(seed)
    masses = (1.0 + rng.random(n)).tolist()
    spring_constants = {}
    for i in range(n):
        for jj in range(i, n):
            spring_constants[(i, jj)] = float(abs(rng.normal(loc=1.0, scale=0.3)))
    return masses, spring_constants


# Choice of convention for the rest of this section - switch to
# build_random_params(N30, seed=0) to exercise the random case instead.
masses_30, spring_constants_30 = build_nearest_neighbor_params(N30)
print(f"N={N30}: {len(spring_constants_30)} (i,j) spring-constant entries, "
      f"{sum(1 for v in spring_constants_30.values() if v != 0.0)} nonzero")

## 2. Hamiltonian and Pauli decomposition via `paulikit`

In [ ]:
####################################################
# Build the N=30 Hamiltonian and decompose it
#  - N=2: dense round-trip reconstruction
#  - N=30: every term checked by direct projection
####################################################

from paulikit.hamiltonian import build_hamiltonian, pad_to_power_of_two
from paulikit.algorithms.fwht import fwht_pauli_terms
from paulikit.pauli_utils import reconstruct_from_terms

# --- Correctness check first, on the already-verified N=2 case,
# before trusting paulikit's decomposition at N=30 ---
_H2 = build_hamiltonian(2, {(0, 0): 1.0, (0, 1): 2.0, (1, 1): 3.0}, [1.0, 2.0], sparse=False)
_H2p, _n2 = pad_to_power_of_two(_H2, sparse=False)
_terms2 = fwht_pauli_terms(_H2p, assume_hermitian=True)
_H2_rebuilt = reconstruct_from_terms(_terms2, _n2)
_max_err_n2 = np.max(np.abs(_H2_rebuilt - _H2p))
print(f"N=2 sanity check: {len(_terms2)} terms, "
      f"reconstruction max abs error = {_max_err_n2:.2e}")
if _max_err_n2 > 1e-10:
    raise RuntimeError("paulikit decomposition does not round-trip at N=2")

# --- Now the real N=30 Hamiltonian ---
H30 = build_hamiltonian(N30, spring_constants_30, masses_30, sparse=True)
H30p, n_qubits_30 = pad_to_power_of_two(H30, sparse=True)
dim_30 = 2 ** n_qubits_30
print(f"N={N30}: padded dimension={dim_30}, n_qubits={n_qubits_30}")

# --- Physics check: the checks below only test the decomposition of
# whatever matrix paulikit returns, so they cannot catch a wrong
# Hamiltonian. H = -[[0, B], [B^T, 0]] reproduces Newton's equations
# only if B B^T = M^{-1/2} F M^{-1/2}, F the stiffness matrix
# (f_jj = sum_k k_jk, f_jk = -k_jk). paulikit < 0.1.4 fails this
# for N >= 3. ---
_F30 = np.zeros((N30, N30))
for (i, j), kappa in spring_constants_30.items():
    _F30[i, i] += kappa
    if i != j:
        _F30[j, j] += kappa
        _F30[i, j] -= kappa
        _F30[j, i] -= kappa
_inv_sqrt_m = np.diag(1.0 / np.sqrt(masses_30))
_B30 = -H30.toarray()[:N30, N30:]
_phys_err_30 = np.max(np.abs(_B30 @ _B30.T - _inv_sqrt_m @ _F30 @ _inv_sqrt_m))
print(f"N={N30} physics check: |B B^T - M^-1/2 F M^-1/2| = {_phys_err_30:.2e}")
if _phys_err_30 > 1e-10:
    raise RuntimeError("Hamiltonian does not reproduce Newton's equations "
                       "(paulikit >= 0.1.4 required)")

terms_30 = fwht_pauli_terms(H30p, assume_hermitian=True)
pauli_labels_30 = list(terms_30.keys())
pauli_coeffs_30 = np.array([terms_30[label] for label in pauli_labels_30], dtype=np.float64)
print(f"N={N30}: {len(pauli_labels_30)} nonzero Pauli terms "
      f"(out of up to 4**{n_qubits_30} = {4 ** n_qubits_30} possible)")

# --- Correctness check at N=30 ---
# Dense round-trip reconstruction (used above for N=2) builds one
# 512x512 matrix per term and takes ~45s here, so instead every
# claimed coefficient is recomputed independently by direct
# projection, c_P = Tr(H P^dagger) / dim, using that a Pauli string
# has one nonzero per row (row r -> column r ^ x_mask). This checks
# all terms, not a sample. Projection only checks the terms paulikit
# reported; Parseval (dim * sum|c|^2 == ||H||_F^2) catches any term
# it dropped. Same method as paulikit's own
# verification/exhaustive_projection.py.


def _popcount(x):
    x = x - ((x >> 1) & 0x5555555555555555)
    x = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
    x = (x + (x >> 4)) & 0x0F0F0F0F0F0F0F0F
    x = x + (x >> 8)
    x = x + (x >> 16)
    x = x + (x >> 32)
    return x & 0x7F


def labels_to_masks(labels, n_qubits):
    """Symplectic (x_mask, z_mask) per label; label[0] = qubit 0 = MSB."""
    chars = np.frombuffer("".join(labels).encode(), dtype=np.uint8)
    chars = chars.reshape(len(labels), n_qubits)
    weights = np.int64(1) << np.arange(n_qubits - 1, -1, -1, dtype=np.int64)
    x_bits = (chars == ord("X")) | (chars == ord("Y"))
    z_bits = (chars == ord("Z")) | (chars == ord("Y"))
    return x_bits @ weights, z_bits @ weights


def verify_by_projection(H, terms, n_qubits, chunk=4096):
    """Return (max coefficient error, relative Parseval gap)."""
    H = H.toarray() if hasattr(H, "toarray") else np.asarray(H)
    dim = 2 ** n_qubits
    labels = list(terms)
    claimed = np.array([terms[k] for k in labels], dtype=np.complex128)
    x, z = labels_to_masks(labels, n_qubits)
    r = np.arange(dim, dtype=np.int64)
    projected = np.empty(len(labels), dtype=np.complex128)
    for s in range(0, len(labels), chunk):
        xs, zs = x[s:s + chunk], z[s:s + chunk]
        cols = r[None, :] ^ xs[:, None]
        sign = 1 - 2 * (_popcount(r[None, :] & zs[:, None]) & 1)
        phase = 1j ** (_popcount(xs & zs) & 3)
        projected[s:s + chunk] = ((H[r[None, :], cols] * sign).sum(axis=1)
                                  * phase / dim)
    frob2 = np.sum(np.abs(H) ** 2)
    parseval_gap = abs(dim * np.sum(np.abs(claimed) ** 2) - frob2) / frob2
    return np.max(np.abs(projected - claimed)), parseval_gap


_proj_err_30, _parseval_gap_30 = verify_by_projection(
    H30p, terms_30, n_qubits_30)
print(f"N=30 projection check, all {len(terms_30)} terms: "
      f"max abs error = {_proj_err_30:.2e}, "
      f"relative Parseval gap = {_parseval_gap_30:.2e}")
if _proj_err_30 > 1e-10 or _parseval_gap_30 > 1e-10:
    raise RuntimeError("paulikit decomposition fails verification at N=30")

## 3. Trotter-repetition bound: closed-form, coefficient-only

<div style="border: 2px solid orange; padding: 10px;">

This bound is a **provably valid but looser** version of the exact
Proposition 16 bound used in N=2 §6.6.1. The exact bound needs a real
operator-2-norm (a singular value decomposition) of a growing "tail"
matrix for every Pauli term; at $N=2$'s 12 terms this is 12 small
SVDs, but at $N=30$'s thousands of terms the same approach needs
thousands of SVDs of $512\times512$ matrices - confirmed directly to
be computationally impractical here (a single such SVD was observed
taking well over a minute under this platform's own BLAS threading
behavior). The closed-form bound below needs no matrices or SVDs at
all - only the Pauli coefficients - but will generally recommend
*more* repetitions than the exact bound would for the same target
accuracy (never fewer: a looser upper bound on the true error can
only be conservative, not optimistic).

</div>

In [ ]:
####################################################
# Closed-form, coefficient-only alpha_comm bound
#  - compute_alpha_comm_2nd_order_closed_form(coeffs)
#  - required_trotter_repetitions_30(t, target_error)
####################################################

import math


def compute_alpha_comm_2nd_order_closed_form(coeffs):
    """Upper bound on Proposition 16's alpha_comm using ONLY the
    Pauli coefficients, no matrices and no operator norms at all.

    Starting point, exactly as in the N=2 notebook's Section 6.6.1:
      alpha_comm = (1/12) sum_g ||[tail_g, [tail_g, H_g]]||
                  + (1/24) sum_g ||[H_g, [H_g, tail_g]]||
    where H_g = c_g * P_g for Pauli string P_g (||P_g|| = 1, since
    every Pauli string is unitary) and tail_g = sum_{g'>g} H_g'.

    Two standard inequalities replace every operator norm with a sum
    over |c_g| alone:
      ||[A, B]|| <= 2 ||A|| ||B||           (sub-multiplicativity)
      ||tail_g|| <= sum_{g'>g} |c_g'|        (triangle inequality, ||P||=1)
    Applying the commutator bound twice (nested commutator) and the
    tail bound once per factor gives, per term g:
      ||[tail_g,[tail_g,H_g]]|| <= 4 * tail_g^2 * |c_g|
      ||[H_g,[H_g,tail_g]]||    <= 4 * |c_g|^2 * tail_g
    Summed over g, this is O(terms) via one running suffix-sum of
    |c_g| - no per-term matrix ever gets built.

    This is strictly looser than the exact bound (verified directly
    against it on the N=2 case: closed-form gives ~4.91 against the
    exact ~0.62, a ~7.9x looser alpha, costing a sqrt(7.9) ~= 2.8x
    larger repetition count for the same target accuracy - a real,
    acceptable cost for being computable at all at N=30's scale).
    """
    coeffs = np.asarray(coeffs, dtype=float)
    abs_c = np.abs(coeffs)
    n = len(abs_c)
    # suffix_sum[g] = sum_{g' > g} |c_g'|
    suffix_sum = np.zeros(n)
    if n > 1:
        suffix_sum[:-1] = np.cumsum(abs_c[::-1])[::-1][1:]
    term1 = np.sum(4.0 * suffix_sum ** 2 * abs_c)
    term2 = np.sum(4.0 * abs_c ** 2 * suffix_sum)
    return term1 / 12.0 + term2 / 24.0


_alpha_30 = compute_alpha_comm_2nd_order_closed_form(pauli_coeffs_30)
print(f"alpha_comm (closed-form, N=30, {len(pauli_coeffs_30)} terms): {_alpha_30:.6g}")


def required_trotter_repetitions_30(t, target_error, alpha=_alpha_30):
    """Repetitions r such that the (looser) closed-form Proposition-16
    bound is <= target_error: alpha * t^3 / r^2 <= target_error."""
    if t == 0:
        return 1
    r = math.sqrt(alpha * abs(t) ** 3 / target_error)
    return max(1, math.ceil(r))


for _t in (0.1, 1.0, 5.0, 10.762932):
    print(
        f"t={_t:9.4f}  repetitions needed for target_error=0.05: "
        f"{required_trotter_repetitions_30(_t, target_error=0.05)}"
    )

## 4. Choosing a demonstration time window

The $N=2$ notebook evolves to $t\approx10.76$, needing
$22{,}058$ repetitions under the (looser) closed-form bound at
$N=30$'s scale - correct, and runnable (about 4 minutes with
`trotterkit` below: about 9.8 ms per repetition, measured on this
machine), but slow enough to be a poor default on every pass through
this notebook. $t=1.0$ needs only $625$ repetitions (about 6
seconds) and is used for the demonstration run below; the cell is written so that setting `t_demo = 10.762932`
reproduces the longer run if wanted. This is a choice of
**demonstration window, not a change in correctness** - the same
Hamiltonian, decomposition, and bound apply at either time.

## 5. Trotterized evolution with `trotterkit`

The $N=2$ notebook uses Classiq's cloud
`synthesize`/`execute` to run the Trotterized circuit. At $N=30$'s
scale this notebook instead evolves the statevector classically with
the repository's `trotterkit` package - a deliberate engineering
choice, not a drop-in substitute for Classiq, made for two concrete
reasons:

1. **Avoiding a cloud round-trip dependency for this demonstration.**
   Classiq's own `synthesize`/`execute` needs network access and
   authentication; a local simulator runs fully offline and
   deterministically.
2. **Performance at this term count.** `trotterkit` applies each
   Pauli exponential to the statevector exactly as the circuit's gate
   would - the same per-term product formula, term by term, forward
   and then backward for the 2nd-order (Strang) step - in a native
   loop over the $2^n$ amplitudes. It uses its C backend by default,
   chosen from a measured comparison of its C, C++ and Rust
   backends; all backends give identical results. It is built into
   the environment by `./bootstrap` (or `make trotterkit`).

Each Pauli-string term $P$ is applied to the statevector via its
closed form $e^{-i\theta P}=\cos\theta\,I-i\sin\theta\,P$ (valid
because $P^2=I$), and $P\psi$ itself is applied
*without ever building the $2^n\times2^n$ matrix*: every Pauli string
has exactly one nonzero entry per row (a bit-flip permutation from any
$X$/$Y$ factors, a sign/phase from any $Y$/$Z$ factors), so $P\psi$
costs $O(\dim)$ instead of $O(\dim^2)$. The same
one-nonzero-per-row structure is what makes the projection check in
§2 cheap. `trotterkit` takes each term as the pair of bit masks
$(x, z)$ built in §2 plus its coefficient.

Before use, the cell below checks `trotterkit` against SciPy's exact
`expm_multiply` evolution at $t=0.5$, with the repetition count the
§3 bound asks for. It also defines `evolve_with_progress`, which runs
a long evolution in chunks and prints a live step counter, elapsed
time and ETA.

In [ ]:
####################################################
# trotterkit evolution, with a progress tracker,
# verified against exact evolution before use
#  - evolve_with_progress(psi, terms=..., dt=..., n_steps=...)
####################################################

import time

try:
    import trotterkit
except ImportError as exc:
    raise ImportError(
        "trotterkit is not installed: run ./bootstrap or "
        "`make trotterkit` from the repository root, then restart "
        "this kernel") from exc
from scipy.sparse.linalg import expm_multiply

# Same term order as pauli_labels_30 / pauli_coeffs_30: the Trotter
# bound in section 3 was computed for exactly this order.
_x_30, _z_30 = labels_to_masks(pauli_labels_30, n_qubits_30)
x_30 = np.ascontiguousarray(_x_30, dtype=np.uint64)
z_30 = np.ascontiguousarray(_z_30, dtype=np.uint64)
coeff_30 = pauli_coeffs_30


def _mmss(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 60}:{seconds % 60:02d}"


def evolve_with_progress(psi, *, terms, dt, n_steps, order=2,
                         backend="c", label="evolve",
                         report_every_s=2.0):
    """Advance psi by n_steps product-formula steps of size dt with
    trotterkit.evolve, in chunks, printing a live progress line.

    Every chunk is a run of steps of the same size dt, so the result
    is the same product formula as one n_steps call (and consecutive
    calls compose): chunking changes only when the progress line is
    printed. The first chunk is about 10 steps, to measure the rate;
    later chunks take about one second each. Returns the new state.
    """
    start = time.perf_counter()
    last_report = start
    done = 0
    chunk = min(10, n_steps)
    while done < n_steps:
        chunk = min(chunk, n_steps - done)
        psi = trotterkit.evolve(psi, t=chunk * dt, steps=chunk,
                                terms=terms, order=order,
                                backend=backend)
        done += chunk
        now = time.perf_counter()
        elapsed = now - start
        rate = done / elapsed
        chunk = max(1, int(rate))
        if done < n_steps and now - last_report >= report_every_s:
            eta = (n_steps - done) / rate
            print(f"{label}: step {done}/{n_steps} "
                  f"({100.0 * done / n_steps:.0f}%) | "
                  f"elapsed {_mmss(elapsed)} | ETA {_mmss(eta)}   ",
                  end="\r", flush=True)
            last_report = now
    print(f"{label}: {n_steps} steps in "
          f"{_mmss(time.perf_counter() - start)}" + " " * 30)
    return psi


# --- Verify trotterkit against exact evolution before using it ---
_t_check = 0.5
_target_check = 0.05
_r_check = required_trotter_repetitions_30(_t_check, _target_check)
_psi0_check = np.zeros(dim_30, dtype=np.complex128)
_psi0_check[0] = 1.0 / np.sqrt(2)
_psi0_check[1] = -1.0 / np.sqrt(2)
_psi_tk = evolve_with_progress(
    _psi0_check, terms=(x_30, z_30, coeff_30), dt=_t_check / _r_check,
    n_steps=_r_check, label="verify")
_psi_exact = expm_multiply(-1j * _t_check * H30p, _psi0_check)
_err_check = np.max(np.abs(_psi_tk - _psi_exact))
print(f"trotterkit vs exact expm_multiply at t={_t_check}, "
      f"{_r_check} steps: max error = {_err_check:.2e} "
      f"(target {_target_check})")
if _err_check > _target_check:
    raise RuntimeError("trotterkit evolution exceeds the Trotter "
                       "error target at t=0.5")


## 6. Running the $N=30$ demonstration

In [ ]:
####################################################
# Run the N=30 Trotterized evolution at the chosen
# demonstration time window
####################################################

import time

t_demo = 1.0  # set to 10.762932 to match the N=2 walkthrough's own
              # time point instead (about 4 min for the chain; the
              # fully coupled random case needs about 15 min at 1.0
              # and about 9 hours at 10.762932)
target_error = 0.05
r_demo = required_trotter_repetitions_30(t_demo, target_error)
dt_demo = t_demo / r_demo
print(f"t={t_demo}, target_error={target_error}: {r_demo} Trotter repetitions")

psi0_30 = np.zeros(dim_30, dtype=np.complex128)
psi0_30[0] = 1.0 / np.sqrt(2)
psi0_30[1] = -1.0 / np.sqrt(2)

_t0 = time.perf_counter()
psi_30 = evolve_with_progress(
    psi0_30, terms=(x_30, z_30, coeff_30), dt=dt_demo,
    n_steps=r_demo, label="N=30 evolution")
_t1 = time.perf_counter()

print(f"ran {r_demo} repetitions in {_t1 - _t0:.1f}s")
print(f"final state norm (should be 1.0): {np.linalg.norm(psi_30):.10f}")


## 7. Next steps

This notebook stops at a verified Hamiltonian, a verified Pauli
decomposition, a verified (if loose) Trotter-repetition bound, and a
verified magnitude-only Trotterized evolution for $N=30$ - the pieces
that do not scale past roughly $N=4$ in the original symbolic/Classiq
approach. **Not yet done, deliberately deferred to the next pass:**

- Decoding `psi_30`'s measurement statistics into the physical
  $|y_i(t)|$ trajectories (the N=2 notebook's `decode_state`,
  generalized from $N=2$ to $N=30$'s qubit layout) and plotting them
  against the analytical solution, the way N=2 §7 does.
- Generalizing N=2 §6.7's Hadamard-test signed-trajectory recovery
  (`prepare_and_evolve_n2` and its driver loop) from 3 qubits to
  $N=30$'s 9 qubits - a real circuit-design task, not a parameter
  change, since that circuit was laid out by hand for the $N=2$ case.
- Performance work is done for this notebook: `trotterkit` provides
  threaded C, C++ and Rust backends, compared by measurement, with C
  as the default.
- A tighter, exact Trotter bound at this scale (the 4th-order-specific
  Proposition J.2 bound used in N=2 §6.6.1 needs a 3-way commuting
  partition of the Hamiltonian's terms, verified by hand for $N=2$'s
  12 terms - finding and verifying an analogous partition for
  thousands of terms is its own piece of work, not attempted here).

## Summary

- `paulikit` replaces SymPy and the brute-force decomposition; at
  $N=30$ (9 qubits) the nearest-neighbor chain has $14{,}080$
  nonzero Pauli terms, each checked by direct projection plus a
  Parseval completeness check, and the Hamiltonian itself is checked
  against Newton's equations.
- A coefficient-only bound replaces the per-term SVDs of the exact
  Trotter bound; it is looser, so it can only over-ask for
  repetitions, never under-ask.
- `trotterkit` applies each Pauli exponential without building a
  matrix, so the $t=1.0$ evolution runs in seconds, verified against
  SciPy's exact `expm_multiply`, with a progress tracker on the
  long loops.

## Further Reading

- A. M. Childs, Y. Su, M. C. Tran, N. Wiebe, S. Zhu, "Theory of Trotter
  Error with Commutator Scaling", Phys. Rev. X 11, 011020 (2021),
  [arXiv:1912.08854](https://arxiv.org/abs/1912.08854)
- R. Babbush et al., "Exponential Quantum Speedup in Simulating
  Coupled Classical Oscillators", Phys. Rev. X 13, 041041 (2023),
  [arXiv:2303.13012](https://arxiv.org/abs/2303.13012)
- `paulikit`: [PyPI](https://pypi.org/project/paulikit/),
  [arXiv:2610.08299](https://arxiv.org/abs/2610.08299)
- Related notebook:
  [`N_coupled_harmonic_oscillators_1_D_N_2.ipynb`](N_coupled_harmonic_oscillators_1_D_N_2.ipynb)